# Homework 04 - TinyValue Implementation

目标：不是读懂 micrograd，而是亲手造一个最小自动求导类。顺序是：前向值 -> 局部反传 -> 拓扑排序 -> 整图 backward。

In [ ]:
from pathlib import Path
import sys, math, random


def find_repo_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'micrograd' / 'engine.py').exists():
            return q
    return p

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def near(a, b, eps=1e-6):
    return abs(a - b) < eps


def todo_guard(values, message='请先填写 TODO，再运行检查。'):
    if any(v is None for v in values):
        print(message)
        return False
    return True


def check_or_todo(condition, message):
    if not condition:
        print(message)
        return False
    return True

## Modify - 先只补加法反传这一行

在完整实现之前，先单独练一行：加法的两个输入都收到 `out.grad`。

In [ ]:
student_add_backward_lines = None
# 期望形状：['self.grad += out.grad', 'other.grad += out.grad']


def qa_check_04_modify():
    if not todo_guard([student_add_backward_lines]): return
    compact = [line.replace(' ', '') for line in student_add_backward_lines]
    assert compact == ['self.grad+=out.grad', 'other.grad+=out.grad']
    print('OK: Modify 通过。')

qa_check_04_modify()

## 完整例子 - 先只看前向对象

这个小对象不会自动求导，只让你确认：`data` 是值，`grad` 先是 0。

In [ ]:
class Box:
    def __init__(self, data):
        self.data = data
        self.grad = 0

x = Box(2.0)
print('x.data =', x.data)
print('x.grad =', x.grad)

## 作业 1-6 - 填 MyTinyValue

先让加法/乘法前向能跑，再补 `_backward`。不要一次想完整个类，每次只修一个测试。

In [ ]:
class MyTinyValue:
    def __init__(self, data, children=(), op=''):
        self.data = data
        self.grad = 0
        self._prev = set(children)
        self._op = op
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, MyTinyValue) else MyTinyValue(other)
        out = MyTinyValue(self.data + other.data, (self, other), '+')

        def _backward():
            # TODO 1: 加法反传，两个输入都收到 out.grad
            pass

        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, MyTinyValue) else MyTinyValue(other)
        out = MyTinyValue(self.data * other.data, (self, other), '*')

        def _backward():
            # TODO 2: 乘法反传：self 收 other.data*out.grad，other 收 self.data*out.grad
            pass

        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = MyTinyValue(self.data ** other, (self,), f'**{other}')

        def _backward():
            # TODO 3: x**n 的导数是 n*x**(n-1)，还要乘 out.grad
            pass

        out._backward = _backward
        return out

    def relu(self):
        out = MyTinyValue(0 if self.data < 0 else self.data, (self,), 'ReLU')

        def _backward():
            # TODO 4: out.data > 0 时传 out.grad，否则传 0
            pass

        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            # TODO 5: 如果访问过就返回；否则递归父节点，最后 append 自己
            pass

        build_topo(self)
        # TODO 6: 最终节点自己对自己导数是 1，然后反向执行 _backward
        pass

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def __repr__(self):
        return f'MyTinyValue(data={self.data}, grad={self.grad})'

In [ ]:
def qa_check_04_forward(ValueClass):
    try:
        a = ValueClass(2.0)
        b = ValueClass(3.0)
        c = a + b
        d = a * b
    except Exception as exc:
        print('前向还不能跑：', type(exc).__name__, exc)
        return
    assert c.data == 5.0, '加法前向 data 错。'
    assert d.data == 6.0, '乘法前向 data 错。'
    assert c._op == '+', '加法节点 _op 应该是 +。'
    assert d._op == '*', '乘法节点 _op 应该是 *。'
    print('OK: 作业 1 前向建图通过。')

qa_check_04_forward(MyTinyValue)

In [ ]:
def qa_check_04_backward(ValueClass):
    try:
        a = ValueClass(2.0)
        b = ValueClass(3.0)
        L = (a * b + a) * 2
        L.backward()
    except Exception as exc:
        print('backward 还不能跑：', type(exc).__name__, exc)
        return

    if not (near(a.grad, 8.0) and near(b.grad, 4.0)):
        print('还没过：L=2(ab+a) 期望 a.grad=8, b.grad=4；实际:', a.grad, b.grad)
        return

    x = ValueClass(3.0)
    y = x * x
    y.backward()
    assert near(x.grad, 6.0), 'L=x*x 时，x 有两条路径，grad 应该是 6。'

    r = ValueClass(-2.0)
    out = r.relu()
    out.backward()
    assert out.data == 0 and r.grad == 0, 'ReLU 负半轴 data=0 且不传梯度。'

    p = ValueClass(3.0)
    q = p ** 2
    q.backward()
    assert near(p.grad, 6.0), 'pow 反传要乘 out.grad。'
    print('OK: 作业 2-6 backward 核心测试通过。')

qa_check_04_backward(MyTinyValue)

<details><summary>Show / Hide 提示</summary>

`build_topo` 的形状是：访问自己 -> 递归父节点 -> append 自己。`backward` 的形状是：`self.grad = 1`，然后 `for node in reversed(topo): node._backward()`。

</details>

## Debug Lab - 修真实错误

In [ ]:
# Debug 1：pow 反传常见错误：只写 n*x**(n-1)，漏乘 out.grad。
student_fixed_pow_line = None
# 期望填字符串：'self.grad += other * self.data ** (other - 1) * out.grad'

# Debug 2：ReLU 常见错误：把 ReLU 的值写成 x>0 时为 1。
student_relu_value_when_positive = None  # ReLU(3) 的值
student_relu_grad_when_positive = None   # ReLU 在 x=3 处的导数


def qa_check_04_debug():
    values = [student_fixed_pow_line, student_relu_value_when_positive, student_relu_grad_when_positive]
    if not todo_guard(values): return
    compact = student_fixed_pow_line.replace(' ', '')
    assert compact == 'self.grad+=other*self.data**(other-1)*out.grad'
    assert student_relu_value_when_positive == 3
    assert student_relu_grad_when_positive == 1
    print('OK: TinyValue Debug Lab 通过。')

qa_check_04_debug()

## 提交清单

- [ ] `qa_check_04_forward` 通过
- [ ] `qa_check_04_backward` 通过
- [ ] Debug Lab 通过
- [ ] 我能解释为什么 `+=`、`out.grad`、拓扑排序缺一不可

<details><summary>Show / Hide 答案</summary>

答案不要一开始看。正确答案应该能由完整例子和 Modify 题一步推出；如果你需要直接看答案，说明前一个台阶还没踩稳。

</details>